# Start here — IoTCom.Net in 10 minutes

IoTCom.Net speaks industrial, navigation, lighting and messaging protocols with one consistent API: **clients** connect, **servers** serve, **publishers** publish, **subscribers** subscribe. Every protocol ships a simulator, so every cell here runs without hardware.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.4.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.4.0-preview.1"

## The shape of every endpoint

All endpoints expose `State` and `StateChanged`, are `IAsyncDisposable`, and are configured with a fluent builder. Transports are swappable: `UseTcp`, `UseSerial`, `UseInMemory`.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Modbus;
using IoTCom.Net.Transports;

var link = new InMemoryTransportListener();
var server = ModbusServer.Create(o => o.ListenInMemory(link));
server.StateChanged += (_, e) => Console.WriteLine($"server: {e.Previous} -> {e.Current}");
await server.StartAsync();

var client = ModbusClient.Create(o => o.UseInMemory(link));
server.Store.HoldingRegisters[0] = 42;
Console.WriteLine($"HR0 = {(await client.ReadHoldingRegistersAsync(0, 1))[0]}");

## See the bytes

Attach a `RecordingTap` to any endpoint to capture every frame — the same data the Gallery and CLI show as a *frame lane*.

In [ ]:
var tap = new RecordingTap();
client.AddTap(tap);
await client.ReadHoldingRegistersAsync(0, 2);
foreach (var f in tap.Snapshot()) Console.WriteLine($"{f.Direction,-8} {HexDump.ToHex(f.Data.Span),-40} {f.Summary}");

## Where next

| Notebook | Topic |
|---|---|
| `industrial/01-modbus` | Modbus master, slave, simulator, Rust engine |
| `transport/02-framing-crc` | CRC catalogue, SLIP, COBS, HDLC |
| `navigation/03-nmea` | GPS/GNSS with NMEA 0183 |
| `messaging/04-mqtt-senml` | MQTT pub/sub with SenML payloads |
| `99-protocol-chooser` | Which protocol for which job |

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*